# সহজ ভাষায় Notebook Guide

এই notebook-এ theory এবং code পাশাপাশি শেখানো হয়েছে। Technical term English-এ থাকবে, আর explanation Bangla-তে—যাতে code-এর language-এর সাথে পরিচিত থেকেও concept সহজে বোঝা যায়।

## কীভাবে ব্যবহার করবেন?

1. Cell উপর থেকে নিচে sequence অনুযায়ী run করুন।
2. Run করার আগে expected output কী হতে পারে লিখে ভাবুন।
3. Output-এর metric, shape এবং visualization explanation-এর সাথে compare করুন।
4. Error হলে import, file path, data shape এবং dependency একে একে check করুন।
5. Notebook শেষে নিজের ভাষায় লিখুন: এটি কোন problem solve করেছে, কীভাবে করেছে এবং limitation কী।

> **Important:** Notebook-এর সব cell successful run হলেই result correct প্রমাণ হয় না; data leakage, wrong assumption এবং misleading metric-ও validate করতে হবে।

# Deep Learning with CNNs: MNIST Digit Classifier

> **Focus Area:** ডিপ লার্নিং (Deep Learning - Convolutional Neural Networks)
> **Topics:** কনভোলিউশনাল লেয়ার (Convolutional Layers); ম্যাক্স পুলিং (Max Pooling); প্যাডিং এবং স্ট্রাইড (Padding & Stride); ইমেজ নরমালইজেশন (Image Normalization)
> **Achievement:** এমন একটি কম্পিউটার ভিশন সিস্টেম তৈরি করা যা মানুষের হাতের লেখা যেকোনো ডিজিট বা সংখ্যা (০-৯) নিখুঁতভাবে চিনে আইডেন্টিফাই করতে পারবে।

---

এখানে আপনার দেওয়া নির্দিষ্ট সেকশনটির একটি চমৎকার এবং সাবলীল **Banglish-English-Bengali** মিক্স রূপান্তর দেওয়া হলো:

---

## 1. Topic: Convolutional Neural Networks (CNNs) 

একটি **Convolutional Neural Network (CNN)** হলো এমন এক ধরণের ডিপ লার্নিং আর্কিটেকচার, যা মূলত গ্রিড-লাইক ডেটা (Grid-like data) — বিশেষ করে **ইমেজ বা ছবি** প্রসেস করার জন্য বিশেষভাবে ডিজাইন করা হয়েছে। সাধারণ নিউরাল নেটওয়ার্কের মতো ছবিকে জোর করে ১D ভেক্টরে ফ্ল্যাটেন (flatten) না করে, CNN ছবির স্পেশাল স্ট্রাকচার বা ত্রিমাত্রিক রূপ (spatial structure) হুবহু ধরে রাখে। এটি করার জন্য মডেলটি ছবির ওপর দিয়ে কিছু লার্নেবল ফিল্টার (learnable filters) স্ক্রল বা স্ক্যান করায়।

**আমরা এই লেকচারে মূলত ৪টি মেইন কম্পোনেন্ট কভার করব:**

* **Convolutional Layers** — ছবির লোকাল ফিচার যেমন: এজ বা বর্ডার (edges), টেক্সচার (textures) এবং আকৃতি (shapes) এক্সট্রাক্ট করে।
* **Max Pooling** — গুরুত্বপূর্ণ ফিচারগুলো অক্ষুণ্ন রেখে ছবির স্পেশাল সাইজ বা ডাইমেনশন কমিয়ে ছোট করে ফেলে (reduce spatial size)।
* **Padding & Stride** — আউটপুট ডাইমেনশন এবং ফিল্টারটি কতটা জায়গা জুড়ে ঘুরবে (coverage) তা পুরোপুরি কন্ট্রোল করে।
* **Image Normalization** — স্টেবল ট্রেনিংয়ের (stable training) জন্য পিক্সেল ভ্যালুগুলোকে একটি নির্দিষ্ট স্কেলে প্রস্তুত করে।

**Project:** MNIST Handwritten Digit Classifier (১০টি ক্লাস: ০-৯ পর্যন্ত হাতের লেখা ডিজিট সনাক্তকরণ)।

## 2. Why It Is Related (কম্পিউটার ভিশনে CNN কেন জরুরি)

### The Problem with Flattening Images 

ঐতিহ্যবাহী ট্র্যাডিশনাল নিউরাল নেটওয়ার্কে (Dense/FC layers) একটি ২D ইমেজকে জোর করে টেনে-হিঁচড়ে ১D ভেক্টরে **ফ্ল্যাটেন (flatten)** করতে হয়। এতে ছবির ভেতরের পিক্সেলগুলোর পারস্পরিক দূরত্ব বা স্পেশাল রিলেশনশিপ (spatial relationships) সম্পূর্ণ ধ্বংস হয়ে যায়।

নিচের টেবিলটি খেয়াল করুন (ধরা যাক হিডেন লেয়ারে ২৫৬টি নিউরন বা ইউনিট আছে):

| Image Size | Flattened Size (1D Vector) | Dense Layer Weights Calculation | Total Parameters |
| --- | --- | --- | --- |
| **28x28 gray** (MNIST) | 784 | $784 \times 256$ | **200,704** |
| **64x64 RGB** | 12,288 | $12,288 \times 256$ | **3.1 million** |
| **224x224 RGB** | 150,528 | $150,528 \times 256$ | **38.5 million** |

**এখানে ২টি মারাত্মক সমস্যা দেখা দেয়:**

1. **Parameter explosion :** ছবির সাইজ সামান্য বাড়লেই প্যারামিটার বা ওয়েটের সংখ্যা কোটিতে গিয়ে ঠেকে। এত বিশাল নেটওয়ার্ক এফিশিয়েন্টলি ট্রেইন করা অসম্ভব।
2. **Spatial destruction :** ১ নম্বর পিক্সেল আর ৭৮৩ নম্বর পিক্সেল ছবির দুই প্রান্তে থাকলেও ফ্ল্যাটেন করার পর নেটওয়ার্ক তাদের সমান দূরত্বে থাকা পিক্সেল হিসেবে ট্রিট করে। ফলে ছবির শেপ বা জ্যামিতিক গঠন বোঝার কোনো উপায় থাকে না।

### Why CNNs Solve This 

CNN মূলত ছবির দুটি মৌলিক বৈশিষ্ট্যের (fundamental properties) চমৎকার সুবিধা নেয়:

1. **Local Connectivity :** একটি নিউরন পুরো ছবির সাথে কানেক্ট না হয়ে শুধুমাত্র একটি ছোট লোকাল অঞ্চলের (যেমন: $3\times3$ পিক্সেল) সাথে কানেক্ট হয়।
2. **Weight Sharing :** আলাদা আলাদা ওয়েট ব্যবহার না করে, একটি সিঙ্গেল ফিল্টার পুরো ছবির ওপর দিয়ে স্লাইড বা স্ক্রল করে যায়। অর্থাৎ, ছবির এক জায়গায় যে ফিল্টারটি এজ (Edge) ডিটেক্ট করছে, সেটিই অন্য জায়গার এজও ডিটেক্ট করতে পারে।

> **Result:** এই ইউনিক মেকানিজমের কারণে CNN প্যারামিটারের সংখ্যা কয়েক মিলিয়ন থেকে কমিয়ে মাত্র **~200K**-তে নিয়ে আসে, অথচ MNIST ডেটাসেটে অনায়াসে **99% এর বেশি Accuracy** অর্জন করে ফেলে!


## 3. How It Works 

### 3.1 The Big Picture Pipeline 

```
Input Image (28x28x1)
    │
    ▼
[Conv2D]  ──→ ছবির এজ (Edges), কার্ভ এবং লাইনগুলো ডিটেক্ট করে
    │
    ▼
[ReLU]    ──→ নেগেটিভ ভ্যালুগুলোকে ০ বানিয়ে নন-লিনিয়ারিটি (Non-linearity) যোগ করে
    │
    ▼
[MaxPool] ──→ সবচেয়ে স্ট্রং সিগন্যালগুলো রেখে ছবির সাইজ অর্ধেক করে ফেলে
    │
    ▼
[Conv2D]  ──→ এবার আরও জটিল আকৃতি, লুপ বা ইন্টারসেকশন ডিটেক্ট করে
    │
    ▼
[ReLU]    ──→ অ্যাক্টিভেশন (Activation)
    │
    ▼
[MaxPool] ──→ সাইজ আরও একবার ছোট করে সংকুচিত করে
    │
    ▼
[Flatten] ──→ ২D ফিচার ম্যাপকে (Feature maps) ১D ভেক্টরে রূপান্তর করে
    │
    ▼
[Dense]   ──→ ক্লাসিফিকেশনের জন্য সব ফিচারকে একসাথে কম্বাইন বা যুক্ত করে
    │
    ▼
[Softmax] ──→ ফাইনাল ১০টি ক্লাসের প্রবাবিলিটি বা আউটপুট দেয় (প্রতি ডিজিটের জন্য একটি)

```

### 3.2 The Core Operation: Convolution 

একটি ছোট ম্যাট্রিক্স বা **Filter** (যেমন: $3\times3$) পুরো ইমেজের ওপর দিয়ে স্লাইড বা **স্ক্রল করে** যায় এবং প্রতিটি পজিশনে একটি **Dot Product** হিসাব করে একটি নতুন **Feature Map** তৈরি করে:

```
Input (5x5)          Filter (3x3)           Feature Map (3x3)
+---+---+---+---+---+   +---+---+---+
| 1 | 1 | 1 | 0 | 0 |   | 1 | 0 | 1 |       +---+---+---+
+---+---+---+---+---+   +---+---+---+       | 4 | 3 | 4 |
| 0 | 1 | 1 | 1 | 0 | x | 0 | 1 | 0 |   =   +---+---+---+
+---+---+---+---+---+   +---+---+---+       | 2 | 4 | 3 |
| 0 | 0 | 1 | 1 | 1 |   | 1 | 0 | 1 |       +---+---+---+
+---+---+---+---+---+   +---+---+---+       | 2 | 3 | 4 |
| 0 | 0 | 1 | 1 | 0 |                       +---+---+---+
+---+---+---+---+---+
| 0 | 1 | 1 | 0 | 0 |
+---+---+---+---+---+

```

**একদম শুরুর পজিশন (0,0)-তে গাণিতিক হিসাবটি কেমন হয়?**


$$(1\times1) + (1\times0) + (1\times1) + (0\times0) + (1\times1) + (1\times0) + (0\times1) + (0\times0) + (1\times1) = 4$$

এভাবে প্রতিটি ফিল্টার ব্যাক-এন্ডে এক একটি নির্দিষ্ট প্যাটার্ন চিনতে শেখে। কোনো একটি নির্দিষ্ট লেয়ারে ৩২টি ভিন্ন ফিল্টার থাকলে আমরা সেখান থেকে ৩২টি ইউনিক ফিচার ম্যাপ পাই।

## 4. Details: How It Works & Valid Points for Using It 

### 4.1 Convolutional Layers — The Feature Detectors 

**How it works :**

* একটি নির্দিষ্ট সাইজের ফিল্টার (যেমন: $3\times3$) একটি নির্দিষ্ট **Stride** বা ধাপ মেনে পুরো ইমেজের ওপর দিয়ে স্লাইড করে।
* প্রতিটি পজিশনে ফিল্টারের ওয়েট (Filter weights) এবং ইমেজের পিক্সেল ভ্যালুর মধ্যে ডট প্রোডাক্ট (Dot product) হিসাব করা হয়।
* এই ক্যালকুলেশনের ফাইনাল রেজাল্টটি আউটপুট **Feature Map**-এর একটি একক ভ্যালু হিসেবে বসে।
* মাল্টিপল ফিল্টার ব্যবহার করে আমরা মাল্টিপল ফিচার ম্যাপ (বা ভিন্ন ভিন্ন চ্যানেল) পেয়ে থাকি।

**Output size formula :** 

$$\text{Output Size} = \left\lfloor \frac{H - F}{S} + 1 \right\rfloor$$


*(এখানে $H = \text{Input Size}$, $F = \text{Filter Size}$, $S = \text{Stride}$ এবং $\lfloor \dots \rfloor$ দিয়ে ফ্লোর বা সর্বনিম্ন পূর্ণসংখ্যা বোঝানো হয়েছে)*

**Valid points for using it :**

* **Parameter efficiency:** একটি $3\times3$ ফিল্টারে মাত্র ৯টি ওয়েট থাকে, যা পুরো ইমেজের সব জায়গায় রি-ইউজ বা বারবার ব্যবহৃত হয়।
* **Translation invariance:** ছবির $(5,5)$ পজিশনে থাকা কোনো ফিচার বা অবজেক্ট যদি সরে গিয়ে $(20,20)$ পজিশনেও বসে, ফিল্টারটি তাকে অনায়াসে ডিটেক্ট করতে পারে।
* **Hierarchical learning:** নেটওয়ার্কের শুরুর লেয়ারগুলো বেসিক এজ (Edges) ডিটেক্ট করে $\rightarrow$ মিডেল লেয়ারগুলো বিভিন্ন আকৃতি বা শেপ (Shapes) ধরে $\rightarrow$ একদম শেষের লেয়ারগুলো সম্পূর্ণ অবজেক্ট (Objects) চিনতে পারে।

---

### 4.2 Padding — Controlling the Border 

**How it works :**

* **Valid padding:** কোনো এক্সট্রা বর্ডার দেওয়া হয় না। ফলে প্রতি লেয়ারে আউটপুট সাইজ $(F-1)$ পরিমাণ সংকুচিত বা ছোট হতে থাকে।
* **Same padding:** ছবির চারপাশ দিয়ে ০ (Zero) এর একটি বর্ডার দেওয়া হয়, যাতে কনভোলিউশন করার পরও আউটপুট সাইজ একদম ইনপুট সাইজের সমান থাকে।

**Valid points for using it :**

* **Same padding** ছবির স্পেশাল ডাইমেনশন ধরে রাখে, যার ফলে আউটপুট সাইজ শূন্য হয়ে যাওয়ার ভয় ছাড়াই অনেক গভীর বা ডিপ (Deep) নেটওয়ার্ক তৈরি করা যায়।
* **Valid padding** কম্পিউটেশনালি সাশ্রয়ী এবং এটি নেটওয়ার্ককে ছবির চারপাশের অপ্রয়োজনীয় অংশ বাদ দিয়ে একদম সেন্ট্রাল ফিচারগুলোতে ফোকাস করতে বাধ্য করে।
* প্যাডিং ব্যবহারের ফলে ইমেজের একদম বর্ডারে বা কোণায় থাকা গুরুত্বপূর্ণ ফিচারগুলোর ইনফরমেশন লস (Information loss) হওয়া থেকে বেঁচে যায়।

---

### 4.3 Stride — Controlling the Step Size 

**How it works :**

* **Stride = 1:** ফিল্টারটি একবারে মাত্র ১ পিক্সেল করে সামনে আগায় (এটি ডিফল্ট মেকানিজম, যেখানে ওভারল্যাপিং কভারেজ বেশি থাকে)।
* **Stride = 2:** ফিল্টারটি একবারে ২ পিক্সেল করে লাফ দেয় (যা আউটপুট সাইজকে সরাসরি অর্ধেক করে ফেলে এবং কম্পিউটেশন ফাস্ট করে)।

**Valid points for using it :**

* কনভোলিউশন লেয়ারে Stride = 1 রাখলে ছবির একদম সূক্ষ্ম বা ফাইন-গ্রেইন্ড স্পেশাল ইনফরমেশনগুলো চমৎকারভাবে সংরক্ষিত থাকে।
* Stride = 2 ব্যবহার করলে (কিংবা এর সাথে পুলিং কম্বাইন করলে) কোনো এক্সট্রা লার্নেবল প্যারামিটার ছাড়াই ডাউনস্যাম্পলিং (Downsampling) করা সম্ভব হয়।
* ⚠️ *সতর্কতা:* নেটওয়ার্কের শুরুর দিকের লেয়ারগুলোতে Stride > 1 দিলে ছোট ছোট গুরুত্বপূর্ণ ফিচারগুলো মিস হয়ে যাওয়ার ঝুঁকি থাকে।

---

### 4.4 Max Pooling — Downsampling with Robustness 

**How it works :**

* ফিচার ম্যাপকে ছোট ছোট নন-ওভারল্যাপিং উইন্ডোতে (যেমন: $2\times2$ অঞ্চল) ভাগ করা হয়।
* প্রতিটি উইন্ডো বা অঞ্চলের পিক্সেলগুলোর মধ্য থেকে শুধুমাত্র **সর্বোচ্চ (Maximum)** ভ্যালুটিকে পিক করা হয়।
* এর ফলে আউটপুট ইমেজের উইডথ এবং হাইট—দুটিই প্রায় অর্ধেক হয়ে যায়।

**Valid points for using it :**

* **Translation invariance:** ইনপুট ইমেজে সামান্য শিফট বা নড়চড় হলেও ম্যাক্স পুলড আউটপুটে কোনো পরিবর্তন আসে না, যা মডেলকে রোবাস্ট করে।
* **Computational efficiency:** এটি ডাইমেনশনকে এক ধাক্কায় ৭৫% পর্যন্ত কমিয়ে ফেলে (Half Width $\times$ Half Height), ফলে প্রসেসিং স্পিড বাড়ে।
* **Overfitting reduction:** পরের লেয়ারগুলোর জন্য প্যারামিটার সংখ্যা অনেক কমে যায়, যা মডেলকে মুখস্থ করা থেকে দূরে রাখে।
* **Feature preservation:** $2\times2$ উইন্ডোর যেকোনো জায়গায় কাঙ্ক্ষিত ফিচারটি থাকলেই ম্যাক্স পুলিং তার স্ট্রং সিগন্যালটি ধরে রাখতে পারে।

---

### 4.5 Image Normalization — Preparing for Training 

**How it works :**

* রিউ ইমেজের পিক্সেল ভ্যালুগুলো সাধারণত ০ থেকে ২৫৫ এর মধ্যে ইন্টিজার (Integer) হয়ে থাকে।
* নরমালইজেশনের মাধ্যমে এই ভ্যালুগুলোকে ২৫৫ দিয়ে ভাগ করে $[0, 1]$ রেঞ্জের ফ্লোটে (Float) নিয়ে আসা হয়।
* অপশনাল হিসেবে অনেক সময় Z-score Normalization করা হয়, যেখানে ডেটার Mean = ০ এবং Std Dev = ১ করা হয়।

```python
# Scaling pixel values to [0, 1] range
X_train = X_train.astype("float32") / 255.0

```

**Valid points for using it :**

* **Gradient stability:** ইনপুট সাইজ ছোট ও নিয়ন্ত্রিত থাকলে ব্যাকপ্রোপাগেশনের (Backpropagation) সময় গ্রাডিয়েন্ট এক্সপ্লোড বা ব্লো-আপ হওয়া বন্ধ হয়।
* **Faster convergence:** Adam বা SGD-র মতো অপ্টিমাইজাররা নরমালইজড ডেটা পেলে অনেক দ্রুত গ্লোবাল মিনিমার দিকে কনভার্জ করতে পারে।
* **Equal feature scales:** কোনো একটি নির্দিষ্ট পিক্সেল বা ফিচারের বিশাল ম্যাগনিচিউড যেন পুরো লস ফাংশনকে একচেটিয়া ডোমিনেট করতে না পারে, তা নিশ্চিত করে।
* **Consistent behavior:** ট্রেইনিং এবং ইনফারেন্স (Inference) — দুই ক্ষেত্রেই একই প্রিপ্রসেসিং মেইনটেইন করলে ডিস্ট্রিবিউশন শিফটের (Distribution shift) প্যারা থাকে না।

## 5. Related Analogy: The Detective's Magnifying Glass

> **কল্পনা করুন একজন ক্রাইম সিন ডিটেকটিভ একটি আতশ কাচ (Magnifying Glass) দিয়ে আঙুলের ছাপ বা ফিঙ্গারপ্রিন্ট পরীক্ষা করছেন।**

| CNN Component | Detective Analogy  |
| --- | --- |
| **Input Image** | মূল ফিঙ্গারপ্রিন্ট কার্ড বা আলামত, যেখানে ১০টি আঙুলের ছাপই আছে। |
| **Convolutional Filter** | গোয়েন্দার হাতের **আতশ কাচ** — যা একবারে পুরো ছবি না দেখে ছোট একটি লোকাল অংশ ফোকাস করে। |
| **Sliding the Filter** | গোয়েন্দা যেভাবে কাচটি পুরো ফিঙ্গারপ্রিন্টের ওপর দিয়ে সিস্টেমেটিকভাবে বাম থেকে ডানে সরায়। |
| **Feature Map** | গোয়েন্দার নোটপ্যাড — যেখানে তিনি দাগিয়ে রাখছেন কোথায় লুপ (Loops), রেখা (Ridges) বা স্পেশাল ঘূর্ণন পাওয়া গেল। |
| **Multiple Filters** | একাধিক গোয়েন্দার একটি টিম — যেখানে একেকজন একেকটি নির্দিষ্ট ক্লু বা প্যাটার্ন খোঁজার দায়িত্বে আছেন। |
| **Max Pooling** | চিফ ইন্সপেক্টর প্রতিটি কোয়াড্রেন্ট বা খণ্ড থেকে শুধু সবচেয়ে স্ট্রং ও মেইন এভিডেন্সগুলো দেখছেন, ছোটখাটো নয়েজ ইগনোর করছেন। |
| **Padding** | ফিঙ্গারপ্রিন্টের চারপাশে একটি ফাঁকা বর্ডার দেওয়া, যাতে গোয়েন্দা কোণায় বা একদম বর্ডারে থাকা আলামতগুলোও মিস না করেন। |
| **Stride** | গোয়েন্দা এক পরীক্ষার পর আতশ কাচটি কতটুকু দূরত্বে সরাচ্ছেন — ছোট পা ফেললে পুঙ্খানুপুঙ্খ স্ক্যান, বড় পা ফেললে দ্রুত ওভারভিউ। |
| **Normalization** | ঝাপসা বা অন্ধকার ছবিটিকে একটি ক্রিস্প ও স্ট্যান্ডার্ডাইজড কনট্রাস্টে রূপান্তর করা, যেন সব গোয়েন্দা একই রকম স্পষ্ট দেখতে পায়। |
| **Deep Layers** | জুনিয়র অফিসাররা দেখছেন সাধারণ রেখা $\to$ সিনিয়র অফিসাররা রেখা মিলিয়ে বানাচ্ছেন জটিল প্যাটার্ন $\to$ পরিশেষে চিফ ইন্সপেক্টর আসামি সনাক্ত করছেন। |

**Why this analogy works :**

* একজন গোয়েন্দা যেভাবে এক নজরে পুরো ফিঙ্গারপ্রিন্ট মুখস্থ করতে পারেন না, ঠিক একইভাবে একটি CNN-ও ছবির প্রতিটি পিক্সেলকে একসাথে প্রতিটি নিউরনের সাথে কানেক্ট করে না।
* দুজনই একটি নির্দিষ্ট **লোকাল ফোকাস** (Local focus বা ছোট উইন্ডো) এবং **সিস্টেমেটিক স্ক্যানিং** (Systematic scanning) ব্যবহার করে।
* দুজনই ধাপে ধাপে একটি **হায়ারার্কিকাল আন্ডারস্ট্যান্ডিং** (Hierarchical understanding) বা স্তরীভূত জ্ঞান তৈরি করে — যেমন: সরল রেখা $\to$ জটিল শেপ $\to$ সম্পূর্ণ অবজেক্ট (অথবা আঙুলের রেখা $\to$ রেখার প্যাটার্ন $\to$ অপরাধীর পরিচয়)।
* ম্যাক্স পুলিং মূলত একটি **সামারি রিপোর্টের (Summary report)** মতো কাজ করে — যেখানে প্রতি লেভেলের রিভিউ শেষে শুধুমাত্র সবচেয়ে শক্তিশালী এবং দরকারি এভিডেন্সটুকুই টিকে থাকে।

---

## 6. Achievement: Build the MNIST Digit Classifier

Now we implement a complete CNN that classifies handwritten digits (0-9) from the MNIST dataset.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# Load & Explore MNIST Data
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Keep the lecture runnable on a normal CPU; the same pipeline scales to all MNIST rows.
X_train, y_train = X_train[:12000], y_train[:12000]
X_test, y_test = X_test[:2000], y_test[:2000]

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape:     {X_test.shape}")
print(f"Number of classes:   {len(np.unique(y_train))}")
print(f"Pixel value range:   {X_train.min()} to {X_train.max()}")

# Visualize sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f"Label: {y_train[i]}", fontsize=12)
    ax.axis('off')
plt.suptitle("Sample MNIST Images", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Preprocessing: Reshape + Normalize

# Reshape to add channel dimension: (N, 28, 28) -> (N, 28, 28, 1)
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

# NORMALIZATION: Scale pixel values from [0, 255] to [0, 1]
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# One-hot encode labels: 5 -> [0,0,0,0,0,1,0,0,0,0]
y_train_cat = to_categorical(y_train, num_classes=10)
y_test_cat = to_categorical(y_test, num_classes=10)

print(f"After reshape:  {X_train.shape}")
print(f"After normalize: min={X_train.min():.2f}, max={X_train.max():.2f}")
print(f"Label example:  {y_train[0]} -> {y_train_cat[0]}")

In [ ]:
# Build the CNN Architecture

model = models.Sequential(name="MNIST_CNN")

# Block 1: Conv + ReLU + MaxPool
model.add(layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu',
                         input_shape=(28, 28, 1), name='conv1'))
# Output: 26x26x32  (because 28 - 3 + 1 = 26)

model.add(layers.MaxPooling2D(pool_size=(2, 2), name='pool1'))
# Output: 13x13x32  (halved by 2x2 pooling)

# Block 2: Conv + ReLU + MaxPool
model.add(layers.Conv2D(filters=64, kernel_size=(3, 3), activation='relu', name='conv2'))
# Output: 11x11x64  (because 13 - 3 + 1 = 11)

model.add(layers.MaxPooling2D(pool_size=(2, 2), name='pool2'))
# Output: 5x5x64  (halved by 2x2 pooling)

# Flatten: 5x5x64 = 1600 values -> 1D vector
model.add(layers.Flatten(name='flatten'))

# Fully-connected classifier
model.add(layers.Dense(128, activation='relu', name='dense1'))
model.add(layers.Dropout(0.5, name='dropout'))
model.add(layers.Dense(10, activation='softmax', name='output'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

In [ ]:
# Train the Model

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3,
        min_lr=1e-6, verbose=1
    ),
]

history = model.fit(
    X_train, y_train_cat,
    batch_size=128,
    epochs=5,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Evaluate & Plot Training History

test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"\nTest Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Test Loss:     {test_loss:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], 'o-', label='Train', color='steelblue')
axes[0].plot(history.history['val_accuracy'], 'o-', label='Validation', color='coral')
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0.85, 1.0])

axes[1].plot(history.history['loss'], 'o-', label='Train', color='steelblue')
axes[1].plot(history.history['val_loss'], 'o-', label='Validation', color='coral')
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize Predictions

y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

fig, axes = plt.subplots(3, 5, figsize=(12, 8))
indices = np.random.choice(len(X_test), 15, replace=False)

for idx, ax in zip(indices, axes.flat):
    ax.imshow(X_test[idx].reshape(28, 28), cmap='gray')
    color = 'green' if y_pred[idx] == y_test[idx] else 'red'
    ax.set_title(f"Pred: {y_pred[idx]} | True: {y_test[idx]}",
                 color=color, fontweight='bold')
    ax.axis('off')

plt.suptitle('Prediction Samples (Green=Correct, Red=Wrong)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix

from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', square=True,
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize Learned Filters (Bonus)

conv1_weights = model.get_layer('conv1').get_weights()[0]

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i < 32:
        ax.imshow(conv1_weights[:, :, 0, i], cmap='viridis')
        ax.set_title(f"F{i+1}", fontsize=8)
    ax.axis('off')

plt.suptitle('First Conv Layer: Learned 3x3 Filters',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Save the Model
model.save("mnist_cnn_model.keras")
print("Model saved to: mnist_cnn_model.keras")